In [7]:
import all_functions
from all_functions import *

# Load the graph & variable list

In [ ]:
edge_index = load_graph('.../edge_index.pt')
list_dir = '.../list.txt'
var_list = get_var_list(list_dir)

# Model deployment & results

In [ ]:
class Main():
    def __init__(self, train_config, env_config, debug=False):

        self.train_config = train_config
        self.env_config = env_config
        self.datestr = None

        train_orig = pd.read_csv('.../train.csv', sep=',', index_col=0)
        test_orig  = pd.read_csv('.../test.csv', sep=',', index_col=0)

        self.train = train_orig.copy()
        self.test  = test_orig.copy()
        
        if 'attack' in self.train.columns:
            train = self.train.drop(columns=['attack'])

        self.var_map = var_list
        
        set_device(env_config['device'])
        self.device = get_device()
        
        edge_index_sets = []
        self.edge_index = edge_index
        edge_index_sets.append(self.edge_index)
        
        train_dataset_indata = construct_data(self.train, var_list, labels=0)
        test_dataset_indata  = construct_data(self.test, var_list, labels=self.test.attack.tolist())

        cfg = {'slide_win': train_config['slide_win'],
               'slide_stride': train_config['slide_stride'],}

        train_dataset = TimeDataset(train_dataset_indata, self.edge_index, mode='train', config=cfg)
        test_dataset  = TimeDataset(test_dataset_indata, self.edge_index, mode='test', config=cfg)

        train_dataloader, val_dataloader = self.get_loaders(train_dataset, train_config['seed'], train_config['batch'], val_ratio = train_config['val_ratio'])
        
        self.train_dataset = train_dataset
        self.test_dataset = test_dataset
        
        self.train_dataloader = train_dataloader
        self.val_dataloader = val_dataloader
        self.test_dataloader = DataLoader(test_dataset, batch_size=train_config['batch'], shuffle=False, num_workers=0)

        self.model = normalModel(edge_index_sets, 
                                 len(var_list), 
                                dim=train_config['dim'], 
                                input_dim=train_config['slide_win'],
                                out_layer_num=train_config['out_layer_num'],
                                out_layer_inter_dim=train_config['out_layer_inter_dim']).to(self.device)

    def get_save_path(self):

        dir_path = self.env_config['save_path']
        
        if self.datestr is None:
            now = datetime.now()
            self.datestr = now.strftime('%m-%d-%H_%M_%S')
        datestr = self.datestr          

        paths = [f'{dir_path}/pretrained/best_{datestr}.pt', 
                 f'{dir_path}/results/{datestr}.csv']

        for path in paths:
            dirname = os.path.dirname(path)
            Path(dirname).mkdir(parents=True, exist_ok=True)

        return paths

    def run(self, target_tick_ppr, q_lo, q_hi, max_iter, tol, k):
        if len(self.env_config['load_model_path']) > 0:
            model_save_path = self.env_config['load_model_path']
            
        else:
            model_save_path = self.get_save_path()[0]

            self.train_log = all_functions.train(self.model, model_save_path, 
                config = self.train_config,
                train_dataloader=self.train_dataloader,
                val_dataloader=self.val_dataloader,
                train_dataset=self.train_dataset)

        self.model.load_state_dict(torch.load(model_save_path, map_location=self.device))        
        best_model = self.model.to(self.device)
        
        _, self.test_result = test(best_model, self.test_dataloader)
        _, self.val_result = test(best_model, self.val_dataloader)
        self.get_score(self.test_result, self.val_result, target_tick_ppr, q_lo, q_hi, max_iter, tol, k)


    def get_loaders(self, train_dataset, seed, batch, val_ratio=0.1):
        dataset_len = int(len(train_dataset))
        train_use_len = int(dataset_len * (1 - val_ratio))
        val_use_len = int(dataset_len * val_ratio)
        
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        val_start_index = random.randrange(train_use_len)
        indices = torch.arange(dataset_len)
        train_sub_indices = torch.cat([indices[:val_start_index], indices[val_start_index+val_use_len:]])
        val_sub_indices = indices[val_start_index:val_start_index+val_use_len]
        
        train_subset = Subset(train_dataset, train_sub_indices)
        val_subset = Subset(train_dataset, val_sub_indices)

        train_dataloader = DataLoader(train_subset, batch_size=batch,shuffle=True)
        val_dataloader = DataLoader(val_subset, batch_size=batch,shuffle=False)

        return train_dataloader, val_dataloader

    def get_score(self, test_result, val_result, target_tick_ppr, q_lo, q_hi, max_iter, tol, k):

        np_test_result = np.array(test_result)
        np_val_result = np.array(val_result)
        test_labels = np_test_result[2, :, 0].tolist()
        
        self.gt_labels = test_labels
        self.test_pred_scores, self.val_pred_scores = get_full_pred_errors(test_result, val_result)
        
        self.sensor_thresholds, self.pred_sensor_labels = get_sensor_anomality_quantile_target_ppr(test_labels, test_result, val_result, target_tick_ppr, q_lo, q_hi, max_iter, tol)
        result_info = get_test_performance_quantile_target_ppr(test_labels, test_result, val_result, target_tick_ppr,q_lo, q_hi, max_iter, tol, k)
        self.result_info = result_info
        
        print('=========================** Result of Timetick Anomaly Detection **============================\n')
        print(f'Macro-F1 score: {result_info[0]}')
        print(f'MCC score: {result_info[1]}\n')
        print(result_info[2])
        return result_info


if __name__ == "__main__":
    
    sys.argv=['']
        
    def set_seed(seed):
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        torch.cuda.manual_seed(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        
    parser = argparse.ArgumentParser()
    parser.add_argument('-batch', help='batch size', type = int, default=16)
    parser.add_argument('-epoch', help='train epoch', type = int, default=100)
    parser.add_argument('-slide_win', help='slide window size', type = int, default=30)
    parser.add_argument('-dim', help='node dimension', type = int, default=64)
    parser.add_argument('-slide_stride', help='slide stride', type = int, default=1)
    parser.add_argument('-save_path', help='save trained model', type = str, default='')
    parser.add_argument('-device', help='cuda / cpu', type = str, default='cuda')
    parser.add_argument('-random_seed', help='random seed', type = int, default=42)
    parser.add_argument('-out_layer_num', help='output size ', type = int, default=1)
    parser.add_argument('-out_layer_inter_dim', help='hidden dim of forecast layer', type = int, default=256)
    parser.add_argument('-decay', help='decay', type = float, default=0)
    parser.add_argument('-val_ratio', help='validation ratio', type = float, default=0.1)
    parser.add_argument('-lr', help='learning rate', type = float, default=0.001)
    parser.add_argument('-load_model_path', help='load pretrained model path', type = str, default='')
    args = parser.parse_args()
    
    train_config = {
        'batch': args.batch,
        'epoch': args.epoch,
        'slide_win': args.slide_win,
        'dim': args.dim,
        'slide_stride': args.slide_stride,
        'seed': args.random_seed,
        'out_layer_num': args.out_layer_num,
        'out_layer_inter_dim': args.out_layer_inter_dim,
        'decay': args.decay,
        'val_ratio': args.val_ratio,
        'lr': args.lr}

    set_seed(train_config['seed'])

    env_config={
        'save_path': args.save_path,
        'device': args.device,
        'load_model_path': args.load_model_path
    }
    
    normal_main = Main(train_config, env_config, debug=False)
    normal_main.run(k=2, target_tick_ppr=0.09, q_lo=0.90, q_hi=0.9999, max_iter=25, tol=0.0001)


## Symptom identification in case of test data containing single fault period.
* Applied for TEP datasets.

In [ ]:
symp_idx, symp_names, counts, freqs, dbg = select_fault_sensors_by_separation(
    pred_sensor_labels=normal_main.pred_sensor_labels,
    gt_tick_labels=normal_main.gt_labels,  
    var_names=normal_main.var_map
)

print("Selected:", symp_names)
print("Counts selected:", counts[symp_idx])
print("All counts sorted:", list(zip(dbg["sorted_names"], dbg["counts_sorted"])))
print("Gaps:", dbg.get("gaps_sorted", None), "cut_k:", dbg.get("cut_k", None))


## Symptom identification in case of test data containing multiple different fault periods.
* Applied for SWaT dataset.

In [ ]:
attack_summary  = summarize_attack_blocks(gt_for_pred=normal_main.gt_labels, pred_sensor_labels=normal_main.pred_sensor_labels, k=15)
row = attack_summary[attack_summary["block_id"] == 17].iloc[0]
s = int(row["start"])
e = int(row["end"])

pred_sensor_labels_sel = normal_main.pred_sensor_labels[:, s:e+1]
gt_labels_sel = normal_main.gt_labels[s:e+1]

symp_idx, symp_names, counts, freqs, dbg = select_fault_sensors_by_separation(
    pred_sensor_labels=pred_sensor_labels_sel,
    gt_tick_labels=gt_labels_sel,  
    var_names=normal_main.var_map)

print("Selected:", symp_names)
print("Counts selected:", counts[symp_idx])
print("All counts sorted:", list(zip(dbg["sorted_names"], dbg["counts_sorted"])))
print("Gaps:", dbg.get("gaps_sorted", None), "cut_k:", dbg.get("cut_k", None))


# Fine-tuning  fault model 

In [ ]:
num_nodes = len(normal_main.var_map) 
sensor_labels = {i: 0 for i in range(num_nodes)}
for i in symp_idx:
    sensor_labels[int(i)] = 1
normal_model = normal_main.model


class faultMain():
    def __init__(self, train_config, env_config, normal_model , debug=False):

        self.train_config = train_config
        self.env_config = env_config
        self.datestr = None

        train_orig  = pd.read_csv('.../test.csv', sep=',', index_col=0)
        test_orig  = pd.read_csv('.../test.csv', sep=',', index_col=0)
        
        ''' 
        # (SWaT data) 
        # Take only selected abnormal period
            raw_s, raw_e = get_selected_attack_window(
                attack_summary,
                block_id=17,
                slide_win=train_config['slide_win']
                    )

            train = train_orig.iloc[raw_s:raw_e+1].reset_index(drop=True)
            test  = test_orig.iloc[raw_s:raw_e+1].reset_index(drop=True)
        '''
        train = train_orig
        test = test_orig
        self.train = train
        self.test  = test
        self.var_map = var_list
    
        set_device(env_config['device'])
        self.device = get_device()

        train_dataset_indata = construct_data(train, var_list, labels=train.attack.tolist())
        test_dataset_indata  = construct_data(test, var_list, labels=test.attack.tolist())

        cfg = {
            'slide_win': train_config['slide_win'],
            'slide_stride': train_config['slide_stride'],
        }

        train_dataset = TimeDataset(train_dataset_indata, edge_index, mode='train', config=cfg)        
        test_dataset  = TimeDataset(test_dataset_indata, edge_index, mode='test', config=cfg)


        train_dataloader, val_dataloader = self.get_loaders(train_dataset, train_config['seed'], train_config['batch'], val_ratio = train_config['val_ratio'])

        self.train_dataset = train_dataset
        self.test_dataset = test_dataset
        self.train_dataloader = train_dataloader
        self.val_dataloader = val_dataloader
        self.test_dataloader = DataLoader(test_dataset, batch_size=train_config['batch'],shuffle=False, num_workers=0)


        edge_index_sets = []
        edge_index_sets.append(edge_index)

        self.normal_model = normal_model
        self.model = faultModel(edge_index_sets, len(var_list), 
                dim=train_config['dim'], 
                input_dim=train_config['slide_win'],
                out_layer_num=train_config['out_layer_num'],
                out_layer_inter_dim=train_config['out_layer_inter_dim'],
                embeddings=normal_model.embedding
            ).to(self.device)
        


    def run(self, normal_main, target_tick_ppr, q_lo, q_hi,max_iter, tol): 
        sd = {k:v for k,v in self.normal_model.state_dict().items()} 
        sd.pop("embedding.weight", None)
        self.model.load_state_dict(sd, strict=False)
                    
        if len(self.env_config['load_model_path']) > 0:
            model_save_path = self.env_config['load_model_path']
        else:
            model_save_path = self.get_save_path()[0]
            self.train_log =train(self.model, model_save_path, 
                config = self.train_config,
                train_dataloader=self.train_dataloader,
                val_dataloader=self.val_dataloader, 
                train_dataset=self.train_dataset)
        
        self.model.load_state_dict(torch.load(model_save_path))
        best_model = self.model.to(self.device)
        self.avg_test_loss, self.test_result = test(best_model, self.test_dataloader)
        _, self.val_result = test(best_model, self.val_dataloader)


    def get_loaders(self, train_dataset, seed, batch, val_ratio=0.1):
        dataset_len = int(len(train_dataset))
        train_use_len = int(dataset_len * (1 - val_ratio))
        val_use_len = int(dataset_len * val_ratio)
        val_start_index = random.randrange(train_use_len)
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        indices = torch.arange(dataset_len)
        train_sub_indices = torch.cat([indices[:val_start_index], indices[val_start_index+val_use_len:]])
        val_sub_indices = indices[val_start_index:val_start_index+val_use_len]

        train_subset = Subset(train_dataset, train_sub_indices)
        val_subset = Subset(train_dataset, val_sub_indices)

        train_dataloader = DataLoader(train_subset, batch_size=batch,shuffle=True)
        val_dataloader = DataLoader(val_subset, batch_size=batch,shuffle=False)

        return train_dataloader, val_dataloader

    def get_save_path(self, feature_name=''):

        dir_path = self.env_config['save_path']
        
        if self.datestr is None:
            now = datetime.now()
            self.datestr = now.strftime('%m-%d-%H_%M_%S')
        datestr = self.datestr          

        paths = [
            f'.../faultModel/pretrained/best_{datestr}.pt']

        for path in paths:
            dirname = os.path.dirname(path)
            Path(dirname).mkdir(parents=True, exist_ok=True)

        return paths



if __name__ == "__main__":
    
    sys.argv=['']
    def set_seed(seed):
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        torch.cuda.manual_seed(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        
    parser = argparse.ArgumentParser()
    parser.add_argument('-batch', help='batch size', type = int, default=16)
    parser.add_argument('-epoch', help='train epoch', type = int, default=100)
    parser.add_argument('-slide_win', help='slide window size', type = int, default=30)
    parser.add_argument('-dim', help='node dimension', type = int, default=64)
    parser.add_argument('-slide_stride', help='slide stride', type = int, default=1)
    parser.add_argument('-save_path', help='save trained model', type = str, default='')
    parser.add_argument('-device', help='cuda / cpu', type = str, default='cpu')
    parser.add_argument('-random_seed', help='random seed', type = int, default=42)
    parser.add_argument('-out_layer_num', help='output size ', type = int, default=1)
    parser.add_argument('-out_layer_inter_dim', help='hidden dim of forecast layer', type = int, default=256)
    parser.add_argument('-decay', help='decay', type = float, default=0)
    parser.add_argument('-val_ratio', help='validation ratio', type = float, default=0.1)
    parser.add_argument('-lr', help='learning rate', type = float, default=0.001)
    parser.add_argument('-load_model_path', help='load pretrained model path', type = str, default='')
    args = parser.parse_args()
    train_config = {
        'batch': args.batch,
        'epoch': args.epoch,
        'slide_win': args.slide_win,
        'dim': args.dim,
        'slide_stride': args.slide_stride,
        'seed': args.random_seed,
        'out_layer_num': args.out_layer_num,
        'out_layer_inter_dim': args.out_layer_inter_dim,
        'decay': args.decay,
        'val_ratio': args.val_ratio,
        'lr': args.lr}
    
    set_seed(train_config['seed'])

    env_config={
        'save_path': args.save_path,
        'device': args.device,
        'load_model_path': args.load_model_path
    }

    fault_main = faultMain(train_config, env_config, normal_model = normal_model, debug=False)
    fault_main.run(normal_main=normal_main, target_tick_ppr=0.09, q_lo=0.90, q_hi=0.9999, max_iter=25, tol=0.0001)


# Attention coefs & change computation

In [ ]:
base_edge_index = normal_main.model.current_gated_edge_index
res = alpha_change_pipeline(
    normal_model=normal_main.model,
    fault_model=fault_main.model,
    X_normal=fault_main.test_dataset.x,
    X_fault=fault_main.test_dataset.x,
    base_edge_index=base_edge_index,
    slide_win=fault_main.train_config['slide_win'],
    fault_start=0,
    layer_idx=0,
    head_reduce="sum",
    proj_agg="sum",
    eps=1e-8)
mean_dif = res["mean_diff"]
ratio = res["ratio"]
pct   = res["pct_change"]
logr  = res["log_ratio"]
base_ei = res["base_edge_index"]

# Saving detection results

In [ ]:
def save_detection_results(
    save_dir,
    result_info,
    symptoms,
    symp_idx,
    sensor_labels,
    fault_model_path,
    res,
    target_tick_ppr,
    q_lo,
    q_hi,
    max_iter,
    tol
):

    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)

    # -------------------------
    # unpack result_info
    # -------------------------
    macro_f1 = result_info[0]
    mcc_coef = result_info[1]
    classification_report_text = result_info[2]


    # -------------------------
    # detection json
    # -------------------------
    detection_data = {
        "fault_model_path": str(fault_model_path),

        "threshold_params": {
            "target_tick_ppr": float(target_tick_ppr),
            "q_lo": float(q_lo),
            "q_hi": float(q_hi),
            "max_iter": int(max_iter),
            "tol": float(tol)
        },

        "metrics": {
            "macro_f1": float(macro_f1),
            "mcc_coef": float(mcc_coef)
        },

        "classification_report": classification_report_text,

        "symptoms": [str(x) for x in symptoms],
        "symptoms_indexes": [int(x) for x in np.asarray(symp_idx).tolist()],
        "sensor_labels": {str(k): int(v) for k, v in sensor_labels.items()},
    }

    with open(save_dir / "detection_results.json", "w", encoding="utf-8") as f:
        json.dump(detection_data, f, indent=2, ensure_ascii=False)

    # -------------------------
    # attention change arrays
    # -------------------------
    np.savez_compressed(
        save_dir / "attention_change.npz",
        mean_diff = np.asarray(res["mean_diff"]),
        ratio=np.asarray(res["ratio"]),
        pct_change=np.asarray(res["pct_change"]),
        log_ratio=np.asarray(res["log_ratio"]),
        base_edge_index=np.asarray(res["base_edge_index"]),
    )

    # -------------------------
    # heatmap
    # -------------------------
    mean_att_diff = np.asarray(res["mean_diff"])
    base_ei = np.asarray(res["base_edge_index"])

    num_nodes = int(base_ei.max()) + 1
    E = base_ei.shape[1]
    sensor_names = [k for k in range(1, num_nodes + 1)]
    diff_mat = np.full((num_nodes, num_nodes), np.nan, dtype=float)
    
    for e in range(E):
        src = int(base_ei[0, e])
        tgt = int(base_ei[1, e])
        diff_mat[src,tgt] = mean_att_diff[e]

    fig, ax = plt.subplots(figsize=(28, 28), dpi=180)
    im = ax.imshow(diff_mat, aspect='equal', vmin=0, vmax=np.nanpercentile(diff_mat, 95))

    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label(" Absolute difference in mean attention", fontsize=18)

    ax.set_xlabel("Target sensor", fontsize=20)
    ax.set_ylabel("Source sensor", fontsize=20)

    ax.set_xticks(np.arange(num_nodes))
    ax.set_yticks(np.arange(num_nodes))
    ax.set_xticklabels(sensor_names, fontsize=12)
    ax.set_yticklabels(sensor_names, fontsize=12)

    ax.set_xticks(np.arange(-0.5, num_nodes, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, num_nodes, 1), minor=True)
    ax.grid(which="minor", color="black", linestyle="-", linewidth=0.3)
    ax.grid(which="major", visible=False)

    val_thresh = np.nanmean(diff_mat)

    for i in range(num_nodes):
        for j in range(num_nodes):
            val = diff_mat[i, j]
            if np.isnan(val):
                continue
            color = "white" if val < val_thresh else "black"
            ax.text(
                j, i, f"{val:.2f}",
                ha="center", va="center",
                fontsize=10, color=color
            )

    plt.title("Absolute Mean Attention Difference per Edge (source → target)", fontsize=20)
    plt.tight_layout()
    plt.savefig(save_dir / "heatmap_abs_diff.png", bbox_inches="tight")
    plt.close(fig)

    print(f"Detection results saved in: {save_dir}")

In [ ]:
save_detection_res_dir = '.../detection_results'

save_detection_results(
    save_dir=save_detection_res_dir,
    result_info=normal_main.result_info,
    symptoms=symp_names,
    symp_idx=symp_idx,
    sensor_labels=sensor_labels,
    fault_model_path=fault_main.get_save_path(),
    res=res,
    target_tick_ppr=0.06,
    q_lo=0.90,
    q_hi=0.9999,
    max_iter=25,
    tol=0.0001
)